# Cheap-First / Expensive-on-Demand — reproducible pipeline

Minimal repo. The `Hierarchical-Model-SNN-main` artifacts are **not** bundled — point to them below.
Run the cells top to bottom.


In [ ]:
import os, sys, subprocess, glob
# EDIT: path to your local artifacts folder (contains FashionMNIST_Experiments, MNIST_Experiments, SVHN_Experiments)
os.environ['SNN_ARTIFACTS'] = os.path.abspath('../Hierarchical-Model-SNN-main/artifacts')
assert os.path.isdir(os.environ['SNN_ARTIFACTS']), 'Set SNN_ARTIFACTS to your artifacts folder'
print('artifacts:', os.environ['SNN_ARTIFACTS'])
PY = sys.executable
def run(*a):
    print('»', *a)
    r = subprocess.run([PY, *a], text=True, capture_output=True)
    print((r.stdout or '')[-1500:])
    if r.returncode: print('ERR:', (r.stderr or '')[-1200:])
    return r.returncode


### 0. (optional) install pinned dependencies


In [ ]:
# !{sys.executable} -m pip install -q -r requirements.txt


### 1. Build residual pools from the artifacts
First run extracts spike-train features (~5-8 min total); cached in `results/_cache/` afterward.


In [ ]:
for ds in ['fashionmnist', 'mnist', 'svhn']:
    run('build_pools.py', ds, '0', '9')
    run('build_pools.py', ds, 'merge')


### 2. E1 (detectors), E2 (conditioning), E3 (cascade cost)


In [ ]:
for ds in ['fashionmnist', 'mnist', 'svhn']:
    for m in ['isi', 'cv']:
        run('real_experiments.py', 'e13', ds, m)
    run('real_experiments.py', 'e2', ds)


### 3. Stage-2 VP/VR (MNIST), fault breakdown, drift, statistics


In [ ]:
run('e4_stage2_vpvr.py')
run('fault_breakdown.py', 'derivable')
run('e5_drift.py')
run('stats_support.py')


### 4. Aggregate report + verification (expect 24/24)


In [ ]:
run('make_report.py')
run('verify_all.py')


### 5. Show the figures and tables


In [ ]:
from IPython.display import Image, display, Markdown
for f in sorted(glob.glob('results/figures/*.png')):
    display(Markdown('**'+os.path.basename(f)+'**')); display(Image(f))


In [ ]:
for t in sorted(glob.glob('results/tables/*.md')):
    display(Markdown(open(t).read()))
